# 01 — Auditoria dos corpora congelados

Confere proveniência, dimensões, splits, grupos e distribuição natural de classes sem alterar os manifests.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


## Ambiente

A captura abaixo registra commit, Python, bibliotecas e disponibilidade do FFmpeg.


In [ ]:
from codeco.notebook_support import environment_snapshot, write_json
environment = environment_snapshot(ROOT)
write_json(environment, REPORTS / "experiment_environment.json")
environment


## Resumo dos manifests

Os manifests versionados, e não as anotações brutas, definem as unidades permitidas no estudo.


In [ ]:
import pandas as pd
from codeco.corpora import CORPUS_MANIFESTS, corpus_summary, load_frozen_manifest

manifests = {name: load_frozen_manifest(name, ROOT) for name in CORPUS_MANIFESTS}
summaries = pd.DataFrame([corpus_summary(df) for df in manifests.values()])
summaries.to_csv(REPORTS / "corpus_summary.csv", index=False)
summaries


## Distribuição natural

As barras contam janelas por espécie antes de qualquer seleção. O eixo logarítmico evidencia a cauda longa.


In [ ]:
import matplotlib.pyplot as plt
from codeco.corpora import class_distribution

fig, axes = plt.subplots(len(manifests), 1, figsize=(12, 5 * len(manifests)))
if len(manifests) == 1:
    axes = [axes]
for ax, (corpus, df) in zip(axes, manifests.items()):
    dist = class_distribution(df)
    ax.bar(range(len(dist)), dist["windows"], color="#3b7d5b")
    ax.set_yscale("log")
    ax.set_title(f"{corpus}: {len(dist)} classes")
    ax.set_ylabel("janelas (log)")
    ax.set_xlabel("classes ordenadas por frequência")
fig.tight_layout()
fig.savefig(REPORTS / "natural_class_distributions.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
distribution_rows = []
for corpus, df in manifests.items():
    dist = class_distribution(df)
    dist.insert(0, "corpus", corpus)
    distribution_rows.append(dist)
natural_distribution = pd.concat(distribution_rows, ignore_index=True)
natural_distribution.to_csv(REPORTS / "natural_class_distributions.csv", index=False)
natural_distribution.groupby("corpus").head(15)
